In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment_family
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_json
from oddball_eeg.loader import load_subjects_grouped, make_subject_summary_table
from oddball_eeg.preprocessing import apply_crop_to_loaded_dataset
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)


set_pandas_display_options()

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10

In [ ]:
# Change only this line when switching datasets.
# EXPERIMENT_NAME = "ds005863_within_subject_main"
EXPERIMENT_NAME = "ds003061_within_subject_main"

experiment_family = get_experiment_family(EXPERIMENT_NAME)
cfg = experiment_family["main"]

# Use the same signal settings as the main benchmark.
epoching_cfg = make_epoching_config_from_experiment(cfg)

# ERP/P300 interpretation window.
P300_TMIN = 0.25
P300_TMAX = 0.50

# Preferred ROI for waveform figures.
ERP_ROI_CANDIDATES = [
    "Pz", "CP1", "CP2", "CP5", "CP6", "Cz", "C3", "C4"
]

OUT_DIR = cfg.artifact_dir("dissertation_figures")
FIG_DIR = cfg.artifact_dir("dissertation_figures", "figures")
TABLE_DIR = cfg.artifact_dir("dissertation_figures", "tables")
TIKZ_DIR = cfg.artifact_dir("dissertation_figures", "tikz_templates")

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "10_thesis_artifacts.ipynb",
        "purpose": "Generate dissertation-ready timing, ERP, topography, and conceptual figure artifacts.",
        "experiment_key": EXPERIMENT_NAME,
        "dataset": cfg.dataset.dataset,
        "task": cfg.dataset.task,
        "signal": cfg.signal.slug(),
        "p300_window": [P300_TMIN, P300_TMAX],
        "artifact_dir": str(OUT_DIR),
        "figure_dir": str(FIG_DIR),
        "table_dir": str(TABLE_DIR),
        "tikz_dir": str(TIKZ_DIR),
    },
)

print("Experiment key :", EXPERIMENT_NAME)
print("Dataset        :", cfg.dataset.dataset)
print("Task           :", cfg.dataset.task)
print("Signal config  :", cfg.signal)
print("Subjects       :", cfg.subjects.subjects)
print("Epoching config:", epoching_cfg)
print("Output dir     :", OUT_DIR)
print("Figure dir     :", FIG_DIR)
print("Table dir      :", TABLE_DIR)
print("TikZ dir       :", TIKZ_DIR)

In [ ]:
data_full = load_subjects_grouped(
    dataset=cfg.dataset.dataset,
    task_name=cfg.dataset.task,
    subjects=cfg.subjects.subjects,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

subject_summary_full = make_subject_summary_table(data_full)
subject_summary_full = round_numeric_columns(subject_summary_full, decimals=4)

display(subject_summary_full)

save_dataframe(
    subject_summary_full,
    TABLE_DIR / "loaded_subject_summary_full_epoch.csv",
)

In [ ]:
data_full = load_subjects_grouped(
    dataset=cfg.dataset.dataset,
    task_name=cfg.dataset.task,
    subjects=cfg.subjects.subjects,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

subject_summary_full = make_subject_summary_table(data_full)
subject_summary_full = round_numeric_columns(subject_summary_full, decimals=4)

display(subject_summary_full)

save_dataframe(
    subject_summary_full,
    TABLE_DIR / "loaded_subject_summary_full_epoch.csv",
)

In [ ]:
from matplotlib.patches import FancyBboxPatch


def add_interval_box(
        ax,
        start: float,
        end: float,
        y: float,
        height: float,
        label: str,
        facecolor: str,
        edgecolor: str,
        alpha: float = 0.25,
        fontsize: int = 12,
):
    patch = FancyBboxPatch(
        (start, y),
        end - start,
        height,
        boxstyle="round,pad=0.01,rounding_size=0.015",
        linewidth=1.4,
        edgecolor=edgecolor,
        facecolor=facecolor,
        alpha=alpha,
    )
    ax.add_patch(patch)

    ax.text(
        (start + end) / 2,
        y + height / 2,
        label,
        ha="center",
        va="center",
        fontsize=fontsize,
    )


def plot_timing_diagram(
        epoch_window: tuple[float, float],
        baseline_window: tuple[float, float],
        crop_window: tuple[float, float],
        p300_window: tuple[float, float],
):
    fig, ax = plt.subplots(figsize=(11, 4.8))

    xmin = epoch_window[0] - 0.05
    xmax = epoch_window[1] + 0.05

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(-0.15, 2.1)

    # Reference lines
    key_times = sorted(set([
        epoch_window[0],
        baseline_window[0],
        baseline_window[1],
        0.0,
        crop_window[0],
        crop_window[1],
        p300_window[0],
        p300_window[1],
        epoch_window[1],
    ]))

    for t in key_times:
        ax.axvline(t, linewidth=0.8, alpha=0.22)

    ax.axvline(0.0, linestyle="--", linewidth=1.4, alpha=0.85)

    # Rows
    y_epoch = 1.25
    y_model = 0.35
    h = 0.45

    add_interval_box(
        ax,
        epoch_window[0],
        epoch_window[1],
        y_epoch,
        h,
        f"Full epoch: {epoch_window[0]:.1f} to {epoch_window[1]:.1f} s",
        facecolor="lightgray",
        edgecolor="black",
        alpha=0.35,
    )

    add_interval_box(
        ax,
        baseline_window[0],
        baseline_window[1],
        y_epoch,
        h,
        "Baseline",
        facecolor="silver",
        edgecolor="black",
        alpha=0.55,
    )

    add_interval_box(
        ax,
        crop_window[0],
        crop_window[1],
        y_model,
        h,
        "Classification crop",
        facecolor="lightgray",
        edgecolor="black",
        alpha=0.35,
    )

    add_interval_box(
        ax,
        p300_window[0],
        p300_window[1],
        y_model,
        h,
        "P300 / ERP window",
        facecolor="silver",
        edgecolor="black",
        alpha=0.55,
    )

    ax.text(
        xmin,
        y_epoch + h / 2,
        "Epoch definition",
        ha="left",
        va="center",
        fontweight="bold",
    )

    ax.text(
        xmin,
        y_model + h / 2,
        "Model / interpretation",
        ha="left",
        va="center",
        fontweight="bold",
    )

    ax.text(
        0.0,
        0.05,
        "Stimulus onset",
        ha="center",
        va="center",
        fontweight="bold",
    )

    ax.set_title("Epoch, baseline, crop, and ERP interpretation windows")
    ax.set_xlabel("Time relative to stimulus onset (s)")
    ax.set_yticks([])
    ax.grid(axis="x", alpha=0.15)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    fig.tight_layout()
    return fig

In [ ]:
fig_timing = plot_timing_diagram(
    epoch_window=(cfg.signal.tmin, cfg.signal.tmax),
    baseline_window=cfg.signal.baseline,
    crop_window=cfg.effective_crop_window(),
    p300_window=(P300_TMIN, P300_TMAX),
)

save_figure(
    fig_timing,
    FIG_DIR / "chapter3_timing_diagram.png",
)

save_figure(
    fig_timing,
    FIG_DIR / "chapter3_timing_diagram.pdf",
)

plt.show()

In [ ]:
def pick_roi_channels(ch_names: list[str], candidates: list[str]) -> list[str]:
    available = []
    name_to_original = {ch.upper(): ch for ch in ch_names}

    for ch in candidates:
        if ch.upper() in name_to_original:
            available.append(name_to_original[ch.upper()])

    if len(available) > 0:
        return available

    # Fallback: use first available channel.
    return [ch_names[0]]


def compute_subject_roi_erp(
        X: np.ndarray,
        y: np.ndarray,
        ch_names: list[str],
        roi_channels: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    roi_idx = [ch_names.index(ch) for ch in roi_channels]

    std = X[y == 0][:, roi_idx, :].mean(axis=(0, 1))
    tgt = X[y == 1][:, roi_idx, :].mean(axis=(0, 1))

    return std, tgt


def compute_grand_average_roi_erp(data, roi_channels: list[str]) -> pd.DataFrame:
    rows = []

    for sub in data.subjects:
        X = data.X_by_sub[sub]
        y = data.y_by_sub[sub]

        std, tgt = compute_subject_roi_erp(
            X=X,
            y=y,
            ch_names=data.ch_names,
            roi_channels=roi_channels,
        )

        for t_idx, t in enumerate(data.times):
            rows.append({
                "subject": sub,
                "time_s": float(t),
                "standard": float(std[t_idx]),
                "target": float(tgt[t_idx]),
                "difference": float(tgt[t_idx] - std[t_idx]),
            })

    df = pd.DataFrame(rows)

    grand = (
        df.groupby("time_s", as_index=False)
        .agg(
            standard_mean=("standard", "mean"),
            standard_std=("standard", "std"),
            target_mean=("target", "mean"),
            target_std=("target", "std"),
            difference_mean=("difference", "mean"),
            difference_std=("difference", "std"),
        )
        .sort_values("time_s")
        .reset_index(drop=True)
    )

    return df, grand

In [ ]:
roi_channels = pick_roi_channels(
    data_full.ch_names,
    ERP_ROI_CANDIDATES,
)

print("ERP ROI channels:", roi_channels)

df_subject_erp, df_grand_erp = compute_grand_average_roi_erp(
    data_full,
    roi_channels=roi_channels,
)

save_dataframe(
    df_subject_erp,
    TABLE_DIR / "subject_roi_erp_waveforms.csv",
)

save_dataframe(
    df_grand_erp,
    TABLE_DIR / "grand_average_roi_erp_waveforms.csv",
)

display(round_numeric_columns(df_grand_erp.head(), 4))

In [ ]:
def plot_grand_average_erp(
        df_grand: pd.DataFrame,
        roi_channels: list[str],
        p300_window: tuple[float, float],
):
    fig, ax = plt.subplots(figsize=(8.5, 4.5))

    ax.plot(
        df_grand["time_s"],
        df_grand["standard_mean"],
        label="Standard",
        linewidth=2,
    )

    ax.plot(
        df_grand["time_s"],
        df_grand["target_mean"],
        label="Target",
        linewidth=2,
    )

    ax.axvline(0.0, linestyle="--", linewidth=1.2)
    ax.axvspan(
        p300_window[0],
        p300_window[1],
        alpha=0.18,
        label=f"{p300_window[0]:.2f}–{p300_window[1]:.2f} s window",
    )

    ax.set_title(
        f"Grand-average ERP waveforms — {cfg.dataset.dataset}, {cfg.dataset.task}\n"
        f"ROI: {', '.join(roi_channels)}"
    )
    ax.set_xlabel("Time relative to stimulus onset (s)")
    ax.set_ylabel("Amplitude")
    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()
    return fig


fig_erp = plot_grand_average_erp(
    df_grand_erp,
    roi_channels=roi_channels,
    p300_window=(P300_TMIN, P300_TMAX),
)

save_figure(
    fig_erp,
    FIG_DIR / "chapter7_grand_average_erp_target_standard.png",
)

save_figure(
    fig_erp,
    FIG_DIR / "chapter7_grand_average_erp_target_standard.pdf",
)

plt.show()

In [ ]:
def plot_difference_waveform(
        df_grand: pd.DataFrame,
        roi_channels: list[str],
        p300_window: tuple[float, float],
):
    fig, ax = plt.subplots(figsize=(8.5, 4.2))

    ax.plot(
        df_grand["time_s"],
        df_grand["difference_mean"],
        label="Target − Standard",
        linewidth=2,
    )

    ax.axhline(0.0, linewidth=1.0)
    ax.axvline(0.0, linestyle="--", linewidth=1.2)
    ax.axvspan(
        p300_window[0],
        p300_window[1],
        alpha=0.18,
        label=f"{p300_window[0]:.2f}–{p300_window[1]:.2f} s window",
    )

    ax.set_title(
        f"Target-minus-standard difference waveform — {cfg.dataset.dataset}, {cfg.dataset.task}\n"
        f"ROI: {', '.join(roi_channels)}"
    )
    ax.set_xlabel("Time relative to stimulus onset (s)")
    ax.set_ylabel("Difference amplitude")
    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()
    return fig


fig_diff = plot_difference_waveform(
    df_grand_erp,
    roi_channels=roi_channels,
    p300_window=(P300_TMIN, P300_TMAX),
)

save_figure(
    fig_diff,
    FIG_DIR / "chapter7_difference_waveform.png",
)

save_figure(
    fig_diff,
    FIG_DIR / "chapter7_difference_waveform.pdf",
)

plt.show()

In [ ]:
def build_mne_info(ch_names: list[str], sfreq: float) -> mne.Info:
    info = mne.create_info(
        ch_names=ch_names,
        sfreq=sfreq,
        ch_types="eeg",
    )
    info.set_montage("standard_1020", on_missing="ignore")
    return info


def compute_subject_p300_diff_topography(
        X: np.ndarray,
        y: np.ndarray,
        times: np.ndarray,
        p300_window: tuple[float, float],
) -> np.ndarray:
    mask = (times >= p300_window[0]) & (times <= p300_window[1])

    if mask.sum() == 0:
        raise ValueError("P300 window is empty.")

    std_topo = X[y == 0][:, :, :][:, :, mask].mean(axis=(0, 2))
    tgt_topo = X[y == 1][:, :, :][:, :, mask].mean(axis=(0, 2))

    return tgt_topo - std_topo


def compute_grand_p300_topography(data, p300_window: tuple[float, float]) -> pd.DataFrame:
    rows = []

    for sub in data.subjects:
        X = data.X_by_sub[sub]
        y = data.y_by_sub[sub]

        topo = compute_subject_p300_diff_topography(
            X=X,
            y=y,
            times=data.times,
            p300_window=p300_window,
        )

        for ch, val in zip(data.ch_names, topo):
            rows.append({
                "subject": sub,
                "channel": ch,
                "p300_difference": float(val),
            })

    df_sub = pd.DataFrame(rows)

    df_grand = (
        df_sub.groupby("channel", as_index=False)
        .agg(
            p300_difference_mean=("p300_difference", "mean"),
            p300_difference_std=("p300_difference", "std"),
        )
        .sort_values("channel")
        .reset_index(drop=True)
    )

    return df_sub, df_grand

In [ ]:
df_subject_topo, df_grand_topo = compute_grand_p300_topography(
    data_full,
    p300_window=(P300_TMIN, P300_TMAX),
)

save_dataframe(
    df_subject_topo,
    TABLE_DIR / "subject_p300_difference_topography.csv",
)

save_dataframe(
    df_grand_topo,
    TABLE_DIR / "grand_p300_difference_topography.csv",
)

display(round_numeric_columns(df_grand_topo.head(), 4))


def plot_p300_topography(
        df_topo: pd.DataFrame,
        ch_names: list[str],
        sfreq: float,
        p300_window: tuple[float, float],
):
    topo_df = (
        df_topo
        .set_index("channel")
        .reindex(ch_names)
        .reset_index()
    )

    values = topo_df["p300_difference_mean"].to_numpy(dtype=float)

    info = build_mne_info(ch_names, sfreq=sfreq)
    picks = valid_topomap_picks(info)

    info_valid = mne.pick_info(info, picks, copy=True)
    values_valid = values[picks]

    fig, ax = plt.subplots(figsize=(5.2, 4.6))

    mne.viz.plot_topomap(
        values_valid,
        info_valid,
        axes=ax,
        show=False,
        contours=6,
    )

    ax.set_title(
        f"Target − standard topography\n"
        f"{p300_window[0]:.2f}–{p300_window[1]:.2f} s, {cfg.dataset.dataset}"
    )

    fig.tight_layout()
    return fig


fig_topo = plot_p300_topography(
    df_grand_topo,
    ch_names=data_full.ch_names,
    sfreq=data_full.sfreq,
    p300_window=(P300_TMIN, P300_TMAX),
)

save_figure(
    fig_topo,
    FIG_DIR / "chapter7_p300_difference_topography.png",
)

save_figure(
    fig_topo,
    FIG_DIR / "chapter7_p300_difference_topography.pdf",
)

plt.show()

In [ ]:
def plot_combined_erp_panel(
        df_grand: pd.DataFrame,
        df_topo: pd.DataFrame,
        ch_names: list[str],
        sfreq: float,
        roi_channels: list[str],
        p300_window: tuple[float, float],
):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

    # Panel A: standard and target ERP
    axes[0].plot(
        df_grand["time_s"],
        df_grand["standard_mean"],
        label="Standard",
        linewidth=2,
    )
    axes[0].plot(
        df_grand["time_s"],
        df_grand["target_mean"],
        label="Target",
        linewidth=2,
    )
    axes[0].axvline(0.0, linestyle="--", linewidth=1.1)
    axes[0].axvspan(p300_window[0], p300_window[1], alpha=0.18)
    axes[0].set_title("Grand-average ERP")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    # Panel B: difference waveform
    axes[1].plot(
        df_grand["time_s"],
        df_grand["difference_mean"],
        linewidth=2,
    )
    axes[1].axhline(0.0, linewidth=1.0)
    axes[1].axvline(0.0, linestyle="--", linewidth=1.1)
    axes[1].axvspan(p300_window[0], p300_window[1], alpha=0.18)
    axes[1].set_title("Target − standard")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Difference")
    axes[1].grid(alpha=0.25)

    # Panel C: topography
    topo_df = (
        df_topo
        .set_index("channel")
        .reindex(ch_names)
        .reset_index()
    )
    values = topo_df["p300_difference_mean"].to_numpy(dtype=float)

    info = build_mne_info(ch_names, sfreq=sfreq)
    picks = valid_topomap_picks(info)

    info_valid = mne.pick_info(info, picks, copy=True)
    values_valid = values[picks]

    mne.viz.plot_topomap(
        values_valid,
        info_valid,
        axes=axes[2],
        show=False,
        contours=6,
    )

    axes[2].set_title(
        f"Topography\n{p300_window[0]:.2f}–{p300_window[1]:.2f} s"
    )

    fig.suptitle(
        f"ERP physiology summary — {cfg.dataset.dataset}, {cfg.dataset.task}\n"
        f"ROI waveform: {', '.join(roi_channels)}",
        y=1.05,
    )

    fig.tight_layout()
    return fig


fig_combined = plot_combined_erp_panel(
    df_grand=df_grand_erp,
    df_topo=df_grand_topo,
    ch_names=data_full.ch_names,
    sfreq=data_full.sfreq,
    roi_channels=roi_channels,
    p300_window=(P300_TMIN, P300_TMAX),
)

save_figure(
    fig_combined,
    FIG_DIR / "chapter7_erp_summary_panel.png",
)

save_figure(
    fig_combined,
    FIG_DIR / "chapter7_erp_summary_panel.pdf",
)

plt.show()

In [ ]:
MAIN_SUMMARY_DIR = cfg.table_dir("summaries")
model_summary_path = MAIN_SUMMARY_DIR / "model_summary_across_subjects.csv"

if model_summary_path.exists():
    df_model_summary = pd.read_csv(model_summary_path)

    heat_df = (
        df_model_summary[["model", "mean_subject_auc"]]
        .copy()
        .sort_values("mean_subject_auc", ascending=False)
        .reset_index(drop=True)
    )

    heat_df["model"] = heat_df["model"].astype(str)

    fig, ax = plt.subplots(figsize=(6.5, 2.7))

    matrix = heat_df[["mean_subject_auc"]].to_numpy()
    im = ax.imshow(matrix, aspect="auto")

    ax.set_yticks(np.arange(len(heat_df)))
    ax.set_yticklabels(heat_df["model"].tolist())
    ax.set_xticks([0])
    ax.set_xticklabels(["Mean subject ROC-AUC"])

    for i, val in enumerate(heat_df["mean_subject_auc"]):
        ax.text(0, i, f"{val:.4f}", ha="center", va="center")

    ax.set_title(
        f"Main grouped benchmark model summary — {cfg.dataset.dataset}, {cfg.dataset.task}"
    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("ROC-AUC")

    fig.tight_layout()

    save_figure(
        fig,
        FIG_DIR / "chapter6_model_summary_heatmap.png",
    )

    plt.show()

    save_dataframe(
        heat_df,
        TABLE_DIR / "chapter6_model_summary_heatmap_values.csv",
    )
else:
    print("Skipping heatmap because model summary does not exist:")
    print(model_summary_path)

In [ ]:
tikz_templates = {
    "benchmark_grid_overview.tex": r"""
\begin{figure}[htbp]
\centering
\begin{tikzpicture}[
    box/.style={draw, rounded corners, minimum width=2.5cm, minimum height=0.9cm, align=center},
    rowlabel/.style={font=\bfseries},
    collabel/.style={font=\bfseries}
]
\node[collabel] at (0,4.2) {};
\node[collabel] at (3,4.2) {LDA};
\node[collabel] at (6,4.2) {Logistic Regression};
\node[collabel] at (9,4.2) {EEGNet};

\node[rowlabel] at (0,3.0) {Raw};
\node[box] at (3,3.0) {};
\node[box] at (6,3.0) {};
\node[box] at (9,3.0) {};

\node[rowlabel] at (0,2.0) {0.1--12 Hz};
\node[box] at (3,2.0) {};
\node[box] at (6,2.0) {};
\node[box] at (9,2.0) {};

\node[rowlabel] at (0,1.0) {1--30 Hz};
\node[box] at (3,1.0) {};
\node[box] at (6,1.0) {};
\node[box] at (9,1.0) {};

\node[rowlabel] at (0,0.0) {ICA-clean};
\node[box] at (3,0.0) {};
\node[box] at (6,0.0) {};
\node[box] at (9,0.0) {};
\end{tikzpicture}
\caption{Conceptual structure of the preprocessing-by-model benchmark grid.}
\label{fig:benchmark_grid_overview}
\end{figure}
""",
    "benchmark_control_logic.tex": r"""
\begin{figure}[htbp]
\centering
\begin{tikzpicture}[
    node distance=1.4cm and 1.8cm,
    box/.style={draw, rounded corners, align=center, minimum width=3.2cm, minimum height=1cm},
    arrow/.style={->, thick}
]
\node[box] (data) {Chronology-aware\\grouped benchmark};
\node[box, below left=of data] (perm) {Permuted-label\\control};
\node[box, below=of data] (prestim) {Pre-stimulus-only\\control};
\node[box, below right=of data] (loso) {Supporting\\LOSO analysis};

\draw[arrow] (data) -- (perm);
\draw[arrow] (data) -- (prestim);
\draw[arrow] (data) -- (loso);
\end{tikzpicture}
\caption{Relationship between the main benchmark and the supporting control and generalization analyses.}
\label{fig:benchmark_control_logic}
\end{figure}
""",
    "project_evolution_simple.tex": r"""
\begin{figure}[htbp]
\centering
\begin{tikzpicture}[
    box/.style={draw, rounded corners, align=center, minimum width=3.2cm, minimum height=1.1cm},
    arrow/.style={->, thick}
]
\node[box] (a) at (0,0) {Sparse autoencoder\\and source-separation\\motivation};
\node[box] (b) at (4.5,0) {Need for stronger\\validation stage};
\node[box] (c) at (9,0) {P300 stimulus\\classification\\benchmark};
\node[box] (d) at (13.5,0) {Leakage-resistant\\model selection\\and interpretation};

\draw[arrow] (a) -- (b);
\draw[arrow] (b) -- (c);
\draw[arrow] (c) -- (d);
\end{tikzpicture}
\caption{Simplified overview of the evolution of the thesis project.}
\label{fig:project_evolution_simple}
\end{figure}
""",
    "preprocessing_routes_simple.tex": r"""
\begin{figure}[htbp]
\centering
\begin{tikzpicture}[
    box/.style={draw, rounded corners, align=center, minimum width=3cm, minimum height=0.9cm},
    arrow/.style={->, thick}
]
\node[box] (raw) at (0,0) {Event-locked EEG trials};
\node[box] (r1) at (4,2) {Raw};
\node[box] (r2) at (4,0.7) {0.1--12 Hz};
\node[box] (r3) at (4,-0.7) {1--30 Hz};
\node[box] (r4) at (4,-2) {ICA-clean};

\draw[arrow] (raw) -- (r1);
\draw[arrow] (raw) -- (r2);
\draw[arrow] (raw) -- (r3);
\draw[arrow] (raw) -- (r4);
\end{tikzpicture}
\caption{Conceptual summary of the main preprocessing routes compared in the benchmark.}
\label{fig:preprocessing_routes_simple}
\end{figure}
""",
}

for filename, content in tikz_templates.items():
    path = TIKZ_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")

print("Saved TikZ templates:")
for path in sorted(TIKZ_DIR.glob("*.tex")):
    print("-", path.name)

In [ ]:
manifest = {
    "experiment_key": EXPERIMENT_NAME,
    "dataset": cfg.dataset.dataset,
    "task": cfg.dataset.task,
    "signal": cfg.signal.slug(),
    "p300_window": [P300_TMIN, P300_TMAX],
    "roi_channels": roi_channels,
    "artifact_dir": str(OUT_DIR),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TABLE_DIR),
    "tikz_dir": str(TIKZ_DIR),
    "figures": sorted([p.name for p in FIG_DIR.glob("*")]),
    "tables": sorted([p.name for p in TABLE_DIR.glob("*")]),
    "tikz_templates": sorted([p.name for p in TIKZ_DIR.glob("*.tex")]),
}

save_json(
    manifest,
    OUT_DIR / "manifest.json",
)

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "10_thesis_artifacts.ipynb",
        "status": "completed",
        "experiment_key": EXPERIMENT_NAME,
        "artifact_manifest": manifest,
    },
)

display(manifest)